# 02 – Cắt crop từng ô đỗ từ bbox

Pipeline chính phân loại **từng ô** (vacant / occupied) nên cần ảnh crop. Notebook cắt mỗi bbox nới rộng **25%** mỗi phía,
resize về **160×160** và lưu theo cấu trúc `ImageFolder`:

```
crops/{train,valid,test}/{vacant,occupied}/{frame}_{box_idx:03d}.jpg
crops/crops.csv   # path, split, camera, date, frame, box_idx, cls
```
Giữ nguyên cách chia train/valid/test của dataset.

**Vì sao lưu rộng hơn vùng dùng để phân loại?** Lúc chạy thật, ô đỗ do người dùng tự vẽ nên rộng/hẹp/lệch khác bbox của PKLot.
Lưu rộng cho phép notebook train cắt ngẫu nhiên để mô hình quen với các cách vẽ khác nhau:

| Dùng ở đâu | Phép biến đổi | Tương đương |
|---|---|---|
| Train | `RandomResizedCrop(128, scale=(0.44, 1.0), ratio=(0.85, 1.18))` | nới 0–25% + dịch ngẫu nhiên |
| Val / Test / chạy thật | `CenterCrop(128)` | đúng vùng bbox + 10% (vì 160 × 1.2 / 1.5 = 128) |

Ô sát mép ảnh: ảnh được nới viền (lặp pixel mép) trước khi cắt để bbox luôn nằm chính giữa crop.

**Sau khi chạy xong trên Kaggle:** *Save Version* → mở tab Output → *New Dataset* để notebook train gắn vào qua *Add Input*.

In [ ]:
import json
import os
from pathlib import Path

HF_REPO = "dronefreak/PKLot"
SPLITS = ["train", "valid", "test"]
IMG_W, IMG_H = 1280, 720
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
DOWNLOAD_DIR = Path("/tmp/pklot")   # không lưu vào output, chỉ dùng trong phiên

def get_hf_token():
    # Có token thì giới hạn tải tính theo tài khoản, không bị chung IP với người khác trên Kaggle (lỗi 429)
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError:   # chạy ngoài Kaggle
        return os.environ.get("HF_TOKEN")
    try:
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print("Không đọc được secret HF_TOKEN:", repr(e))
        return os.environ.get("HF_TOKEN")

def write_yolo_labels(data_root: Path):
    # Tạo labels/*.txt từ metadata.jsonl (bbox COCO pixel → YOLO chuẩn hóa), khớp 100% file label gốc
    for split in SPLITS:
        out = data_root / "labels" / split
        out.mkdir(parents=True, exist_ok=True)
        for line in (data_root / "images" / split / "metadata.jsonl").read_text().splitlines():
            rec = json.loads(line)
            rows = [f"{c} {(x + w / 2) / IMG_W:.6f} {(y + h / 2) / IMG_H:.6f} {w / IMG_W:.6f} {h / IMG_H:.6f}"
                    for (x, y, w, h), c in zip(rec["objects"]["bbox"], rec["objects"]["categories"])]
            (out / f"{Path(rec['file_name']).stem}.txt").write_text("\n".join(rows) + "\n")

def download_pklot(dest: Path) -> Path:
    from huggingface_hub import snapshot_download   # cần bật Internet trong Settings
    token = get_hf_token()
    if not token:
        # Dừng ngay thay vì tải ẩn danh hàng giờ vì bị giới hạn (HTTP 429)
        raise RuntimeError(
            "Chưa có HF_TOKEN. Trên Kaggle: Add-ons → Secrets → thêm secret tên đúng 'HF_TOKEN' "
            "và ĐÁNH DẤU ô bên cạnh để gắn vào notebook này, rồi chạy lại. "
            "Hoặc Add Input dataset PKLot đã lưu để khỏi phải tải.")
    # Chỉ tải ảnh + metadata.jsonl; label tự tạo lại → giảm một nửa số request.
    # Chạy lại cell sẽ bỏ qua các file đã tải xong.
    snapshot_download(HF_REPO, repo_type="dataset", local_dir=dest, token=token, max_workers=8,
                      allow_patterns=["data/data.yaml", "data/images/*"])
    write_yolo_labels(dest / "data")
    (dest / "data" / ".complete").touch()
    return dest / "data"

def find_data_root() -> Path:
    # 1) Dataset đã gắn qua Add Input  2) đã tải xong trước đó  3) tải từ Hugging Face
    for depth in range(1, 6):   # không dùng "**" để tránh duyệt hết hàng chục nghìn ảnh
        for yaml_path in Path("/kaggle/input").glob("*/" * depth + "data.yaml"):
            if (yaml_path.parent / "labels").is_dir():
                return yaml_path.parent
    if (DOWNLOAD_DIR / "data" / ".complete").exists():
        return DOWNLOAD_DIR / "data"
    return download_pklot(DOWNLOAD_DIR)

DATA_ROOT = find_data_root()
print("DATA_ROOT =", DATA_ROOT)

In [ ]:
from concurrent.futures import ThreadPoolExecutor

import cv2
import numpy as np
from tqdm.auto import tqdm

SAVE_PAD = 0.25            # lưu bbox nới 25% mỗi phía (cho augment cắt lệch khi train)
SAVE_SIZE = 160            # 160 × (1 + 2·0.10) / (1 + 2·0.25) = 128 → CenterCrop(128) = bbox + 10%
JPEG_QUALITY = 90
# Ảnh chụp cách nhau ~5 phút nên các frame liền kề gần giống nhau; lấy 1/2 frame của train để giảm dung lượng.
FRAME_STRIDE = {"train": 2, "valid": 1, "test": 1}
OUT_DIR = WORK_DIR / "crops"

import pandas as pd

CLASS_NAMES = {0: "vacant", 1: "occupied"}

def load_boxes(data_root: Path) -> pd.DataFrame:
    # Đọc toàn bộ label YOLO thành bảng, mỗi dòng là một ô đỗ (bbox theo pixel)
    rows = []
    for split in SPLITS:
        for txt in sorted((data_root / "labels" / split).glob("*.txt")):
            camera, date, time = txt.stem.split("_", 2)
            for idx, line in enumerate(txt.read_text().split("\n")):
                if not line.strip():
                    continue
                cls, xc, yc, w, h = line.split()
                xc, yc, w, h = float(xc) * IMG_W, float(yc) * IMG_H, float(w) * IMG_W, float(h) * IMG_H
                rows.append((split, txt.stem, camera, date, time, idx, int(cls),
                             xc - w / 2, yc - h / 2, w, h))
    return pd.DataFrame(rows, columns=["split", "frame", "camera", "date", "time", "box_idx", "cls",
                                       "x", "y", "w", "h"])

boxes = load_boxes(DATA_ROOT)
print(f"{boxes['frame'].nunique():,} ảnh, {len(boxes):,} ô đỗ")

In [ ]:
keep = boxes.groupby("split")["frame"].transform(
    lambda s: s.isin(sorted(s.unique())[::FRAME_STRIDE[s.name]]))
boxes = boxes[keep].reset_index(drop=True)
print(f"Sau khi lấy mẫu frame: {boxes['frame'].nunique():,} ảnh, {len(boxes):,} crop")
print(pd.crosstab(boxes["split"], boxes["cls"].map(CLASS_NAMES)))

In [ ]:
for split in SPLITS:
    for name in CLASS_NAMES.values():
        (OUT_DIR / split / name).mkdir(parents=True, exist_ok=True)

# Nới viền ảnh đủ rộng để vùng cắt không bao giờ bị cụt ở mép → bbox luôn nằm chính giữa crop
BORDER = int(np.ceil(SAVE_PAD * max(boxes["w"].max(), boxes["h"].max()))) + 2

def crop_frame(item):
    (split, frame), g = item
    img = cv2.imread(str(DATA_ROOT / "images" / split / f"{frame}.jpg"))
    img = cv2.copyMakeBorder(img, BORDER, BORDER, BORDER, BORDER, cv2.BORDER_REPLICATE)
    paths = []
    for b in g.itertuples():
        x0 = round(b.x - SAVE_PAD * b.w) + BORDER; x1 = round(b.x + b.w * (1 + SAVE_PAD)) + BORDER
        y0 = round(b.y - SAVE_PAD * b.h) + BORDER; y1 = round(b.y + b.h * (1 + SAVE_PAD)) + BORDER
        crop = cv2.resize(img[y0:y1, x0:x1], (SAVE_SIZE, SAVE_SIZE), interpolation=cv2.INTER_AREA)
        rel = Path(split) / CLASS_NAMES[b.cls] / f"{frame}_{b.box_idx:03d}.jpg"
        cv2.imwrite(str(OUT_DIR / rel), crop, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
        paths.append(rel.as_posix())
    return g.index, paths

groups = list(boxes.groupby(["split", "frame"]))
boxes["path"] = None
with ThreadPoolExecutor(max_workers=os.cpu_count()) as pool:   # OpenCV nhả GIL nên thread chạy song song được
    for idx, paths in tqdm(pool.map(crop_frame, groups), total=len(groups)):
        boxes.loc[idx, "path"] = paths

cols = ["path", "split", "camera", "date", "frame", "box_idx", "cls"]
boxes[cols].to_csv(OUT_DIR / "crops.csv", index=False)
print("Đã lưu", len(boxes), "crop vào", OUT_DIR)

## Kiểm tra kết quả

In [ ]:
import matplotlib.pyplot as plt

crops = pd.read_csv(OUT_DIR / "crops.csv")
assert all((OUT_DIR / p).exists() for p in crops["path"].sample(min(500, len(crops)), random_state=0))

# Khung vàng = vùng CenterCrop(128) dùng khi đánh giá / chạy thật (bbox + 10%)
c0 = (SAVE_SIZE - 128) // 2
fig, axes = plt.subplots(2, 10, figsize=(16, 3.6))
for row, (cls, name) in enumerate(CLASS_NAMES.items()):
    for ax, p in zip(axes[row], crops[crops["cls"] == cls]["path"].sample(10, random_state=1)):
        ax.imshow(cv2.cvtColor(cv2.imread(str(OUT_DIR / p)), cv2.COLOR_BGR2RGB))
        ax.add_patch(plt.Rectangle((c0, c0), 128, 128, fill=False, edgecolor="yellow", linewidth=1))
        ax.axis("off")
    axes[row][0].set_title(name, loc="left")
plt.tight_layout()
plt.savefig(WORK_DIR / "crop_samples.png", dpi=120)
plt.show()

size_gb = sum(f.stat().st_size for f in OUT_DIR.rglob("*.jpg")) / 1e9
print(f"Dung lượng crops: {size_gb:.2f} GB")